# Brand recommender

This notebook has a small manual check for contextual bandit recommendations. The feedback table says which example brands are loved, maybe, or no thanks, then the notebook ranks the next brands from those reactions.


## Setup

Repo-root or step-folder paths.


In [1]:
from pathlib import Path

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "step_5_recommender" else CWD
STEP_DIR = ROOT / "step_5_recommender"

source_path = ROOT / "helpers" / "source_helpers.py"
exec(compile(source_path.read_text(encoding="utf-8"), str(source_path), "exec"), globals())

helper_path = STEP_DIR / "helpers" / "recommender_helpers.py"
exec(compile(helper_path.read_text(encoding="utf-8"), str(helper_path), "exec"), globals())


All embedded brands: 1,689
Recommender catalog: 1,689 brands (text embeddings)
Embedding dimensions: 768


## Checks

Recommender files used by the manual check.


In [2]:
from IPython.display import display

embedding_metadata, embedding_matrix, resolved_embedding_dir = load_embeddings(ROOT)
recommender_artifacts = pd.DataFrame([
    {"artifact": "embedding metadata", "path": str(resolved_embedding_dir.relative_to(ROOT) / "brand_metadata.csv"), "rows": len(embedding_metadata)},
    {"artifact": "embedding matrix", "path": str(resolved_embedding_dir.relative_to(ROOT) / "brand_embeddings.npz"), "rows": embedding_matrix.shape[0]},
    {"artifact": f"recommender catalog ({catalog_mode})", "path": str(resolved_embedding_dir.relative_to(ROOT) / "brand_metadata.csv"), "rows": len(catalog)},
])

save_table(recommender_artifacts, "recommender_artifacts", ROOT)

display(recommender_artifacts)


,artifact,path,rows
0,embedding metadata,step_2_text_embeddings/brand_metadata.csv,1689
1,embedding matrix,step_2_text_embeddings/brand_embeddings.npz,1689
2,recommender catalog (text embeddings),step_2_text_embeddings/brand_metadata.csv,1689


## Manual check

This example likes playful, colourful clothes brands and says no thanks to quieter minimal brands, so the recommendation table should move toward that playful direction.


In [3]:
TEST_CATEGORY = "clothes"

MANUAL_FEEDBACK = pd.DataFrame([
    {"brand_name": "Ganni", "reaction": "Love it"},
    {"brand_name": "Miu Miu", "reaction": "Love it"},
    {"brand_name": "Stine Goya", "reaction": "Love it"},
    {"brand_name": "Sandy Liang", "reaction": "Love it"},
    {"brand_name": "& Other Stories", "reaction": "Maybe"},
    {"brand_name": "Rixo", "reaction": "Maybe"},
    {"brand_name": "Reformation", "reaction": "Maybe"},
    {"brand_name": "The Row", "reaction": "No, thanks"},
    {"brand_name": "Jil Sander", "reaction": "No, thanks"},
    {"brand_name": "Lemaire", "reaction": "No, thanks"},
])

manual_catalog = category_catalog(TEST_CATEGORY)
manual_embeddings = category_embeddings(TEST_CATEGORY)

unknown_reactions = sorted(set(MANUAL_FEEDBACK['reaction']) - set(REWARD_VALUES))
if unknown_reactions:
    raise ValueError(f"Unknown reactions: {unknown_reactions}")

manual_feedback_table = MANUAL_FEEDBACK.merge(
    manual_catalog[['brand_name', 'category', 'aesthetic_keywords', 'silhouettes', 'materials', 'palette']],
    on='brand_name',
    how='left',
)
if manual_feedback_table['category'].isna().any():
    missing = manual_feedback_table.loc[manual_feedback_table['category'].isna(), 'brand_name'].tolist()
    raise ValueError(f"Manual feedback brands are missing from {TEST_CATEGORY}: {missing}")

save_table(manual_feedback_table, "recommender_manual_feedback", ROOT)
manual_feedback_table


,brand_name,reaction,category,aesthetic_keywords,silhouettes,materials,palette
0,Ganni,Love it,clothes,"playful, contemporary, bold, joyful","ruffled dresses, relaxed tailoring, statement ...","organic cotton, recycled polyester, denim","leopard, brights, black, floral, ecru"
1,Miu Miu,Love it,clothes,"playful, offbeat, retro, coquettish, refined","micro-mini cuts, retro tailoring, embellished ...","wool, satin, leather, crystal","soft tones, black, brights, print"
2,Stine Goya,Love it,clothes,"playful, colourful, artful, joyful","print dresses, statement tailoring, bold knits","recycled fabrics, viscose, jacquard","bold print, brights, pastel"
3,Sandy Liang,Love it,clothes,"playful, nostalgic, coquettish, downtown, sweet","fleece outerwear, bow details, easy separates","fleece, cotton, knit","soft pastels, brights, black, print"
4,& Other Stories,Maybe,clothes,"curated, feminine, polished, trend-aware, mode...","print dresses, tailored separates, soft blouse...","cotton, silk, wool, leather","ecru, black, soft pastels, print"
5,Rixo,Maybe,clothes,"vintage, dreamy, soft-romantic, nostalgic, pai...","bias-cut dresses, midi tea dresses, ruffled bl...","silk, chiffon, printed crepe, lace","painterly florals, jewel tones, dusty pastels,..."
6,Reformation,Maybe,clothes,"feminine, sustainable, retro, sexy, effortless","bias slip dresses, fitted midi dresses, vintag...","deadstock fabrics, viscose, linen, Tencel","florals, black, ivory, jewel tones"
7,The Row,"No, thanks",clothes,"restrained refinement, understated, refined, c...","oversized coats, fluid suiting, relaxed cashme...","cashmere, vicuna, fine wool, silk, soft leather","ivory, oat, taupe, charcoal, black"
8,Jil Sander,"No, thanks",clothes,"minimal, architectural, refined, quiet, precis...","clean lines, structured shoulders, elongated c...","wool, cashmere, fine cotton, silk, leather","ivory, stone, black, navy, camel"
9,Lemaire,"No, thanks",clothes,"quiet, soft, neutral, considered, daily, under...","rounded shoulders, loose trousers, soft tailor...","cotton, wool, silk, leather, technical blends","taupe, sand, clay, dark brown, ink"


In [4]:
name_to_idx = {name: i for i, name in enumerate(manual_catalog['brand_name'])}
manual_events = [
    (name_to_idx[row.brand_name], REWARD_VALUES[row.reaction])
    for row in manual_feedback_table.itertuples(index=False)
]
manual_seen = [idx for idx, _ in manual_events]

manual_bandit = fit_bandit(manual_events, manual_embeddings, alpha=0.35)
manual_recs = rank_brands(manual_bandit, manual_embeddings, manual_catalog, seen=manual_seen, top_k=12)

save_table(manual_recs, "recommender_manual_recommendations", ROOT)
manual_recs


,rank,catalog_index,record_id,brand_name,category,aesthetic_keywords,silhouettes,materials,palette,ucb_score,exploit,explore
0,1,210,clothes::damson_madder,Damson Madder,clothes,"playful, sustainable, contemporary, print, fun","print dresses, volume sleeves, easy separates","organic cotton, recycled fabrics","bold print, ecru, brights, floral",0.7440,0.5859,0.1581
1,2,697,clothes::tyler_mcgillivary,Tyler McGillivary,clothes,"playful, colourful, artsy, y2k, print-led","printed mesh tops, mini dresses, matching sets...","mesh, jersey, cotton, knit blends","rainbow brights, black, graphic prints",0.7437,0.5783,0.1654
2,3,196,clothes::collina_strada,Collina Strada,clothes,"playful, sustainable, joyful, upcycled, experi...","ruched dresses, print separates, draped pieces","deadstock, recycled fabrics, knit","bold print, brights, tie-dye",0.7407,0.5805,0.1602
3,4,691,clothes::traffic_people,Traffic People,clothes,"retro, whimsical, print-led, feminine, playful","print dresses, vintage-cut separates, easy pieces","viscose, crepe","retro print, brights, black",0.7402,0.5673,0.1729
4,5,250,clothes::fashion_brand_company,Fashion Brand Company,clothes,"surreal, ironic, playful, camp, internet-led","novelty dresses, graphic tees, cut-out sets, a...","cotton jersey, mesh, stretch fabric","black, white, brights, novelty prints",0.7347,0.5346,0.2001
5,6,413,clothes::made_by_minga,Made By Minga,clothes,"y2k, playful, graphic, colourful, internet-led","baby tees, mini skirts, cardigans, graphic knits","cotton jersey, acrylic knits, denim","pastels, red, black, novelty graphics",0.7336,0.5607,0.1729
6,7,475,clothes::msgm,MSGM,clothes,"playful, bold, youthful, print-led, energetic","print dresses, graphic separates, easy tailoring","cotton, jersey, jacquard","bold print, brights, black",0.7335,0.5774,0.1560
7,8,233,clothes::doodlage,Doodlage,clothes,"sustainable, upcycled, conscious, playful, craft","patchwork separates, easy dresses, print pieces","upcycled fabrics, organic cotton","mixed print, ecru, brights",0.7331,0.5622,0.1710
8,9,512,clothes::olivia_rubin,Olivia Rubin,clothes,"playful, colourful, party, feminine, print","sequin dresses, rainbow knits, print separates","sequin, viscose, knit","rainbow, pastel, brights, print",0.7318,0.5558,0.1760
9,10,67,clothes::anna_sui,Anna Sui,clothes,"bohemian, vintage, eclectic, romantic, rock","baby-doll dresses, print mixes, ruffles, retro...","printed chiffon, lace, velvet, crochet","jewel tones, print, black, purple",0.7308,0.5661,0.1647
